In [ ]:
ROLLING_WINDOWS = [4, 12]
ROLLING_MIN_PERIODS = 2
EMA_FAST_SPAN = 4
EMA_SLOW_SPAN = 12
SBC_WINDOW = 52
INTER_DEMAND_WINDOW = 52
VOLUME_WINDOW = 52
VOLUME_MIN_PERIODS = 5
ZERO_RATIO_WINDOW = 12
RECENT_WINDOW = 4
RECENT_SHIFT = 4
CROSTON_ALPHA = 0.1
SBC_ADI_THRESHOLD = 4.0
SBC_CV2_THRESHOLD = 0.3
FOURIER_PERIOD = 52

In [ ]:
FEATURES = [
    'zero_ratio_12', 'demand_rate', 'q75_nonzero_demand', 'mean_nonzero_demand',
    'std_nonzero_demand', 'median_nonzero_demand', 'croston_alpha_demand',
    'croston_alpha_interval', 'croston_forecast', 'mean_demand_interval',
    'rolling_mean_12', 'rolling_std_12', 'ema_slow', 'ema_fast', 'macd',
    'sales_ratio', 'rolling_mean_4', 'rolling_std_4', 'recent_trend_4',
    'sbc_category', 'sin_1', 'sin_2', 'cos_1', 'adi', 'cv2',
    'cos_2', 'lag_1', 'weeks_since_last_demand', 'history_weeks',
]

In [ ]:
import pandas as pd
from google.colab import userdata
import numpy as np

data_path = userdata.get('DATA_PATH')
df_edit = pd.read_csv(data_path)

df_edit['week'] = pd.to_datetime(df_edit['week'])

In [ ]:
def _croston(y, alpha):
    """y: (T, P) ürün ömrü içi seri (ömür öncesi NaN). Çıktı t satırı yalnızca y[:t] kullanır."""
    T, P = y.shape
    d_est = np.full(P, np.nan)
    i_est = np.full(P, np.nan)
    prev = np.full(P, np.nan)
    out_d = np.full((T, P), np.nan)
    out_i = np.full((T, P), np.nan)
    for t in range(1, T):
        yj = y[t - 1]
        hit = np.nan_to_num(yj, nan=0.0) > 0
        d_est = np.where(hit, np.where(np.isnan(d_est), yj, alpha * yj + (1 - alpha) * d_est), d_est)
        upd_i = hit & ~np.isnan(prev)
        gap = (t - 1) - prev
        i_est = np.where(upd_i, np.where(np.isnan(i_est), gap, alpha * gap + (1 - alpha) * i_est), i_est)
        prev = np.where(hit, t - 1, prev)
        out_d[t] = d_est
        out_i[t] = i_est
    return out_d, out_i

In [ ]:
def build_features(df):
    """Her feature, t haftası için yalnızca t-1 ve öncesindeki satışlardan hesaplanır.
    Ürün ilk satış haftasında başlar; ilk satıştan önceki sıfırlar ürün ömrüne dahil değildir."""
    wide = df.pivot(index='week', columns='product', values='quantity').sort_index().astype('float64')
    T, P = wide.shape
    t_idx = np.arange(T)[:, None]

    positive = wide.values > 0
    has_sale = positive.any(axis=0)
    first = positive.argmax(axis=0)
    life = (t_idx >= first[None, :]) & has_sale[None, :]

    y = wide.where(life)
    lag1 = y.shift(1)
    f = {'lag_1': lag1}

    # rolling
    for w in ROLLING_WINDOWS:
        r = lag1.rolling(w, min_periods=ROLLING_MIN_PERIODS)
        f[f'rolling_mean_{w}'] = r.mean()
        f[f'rolling_std_{w}'] = r.std()

    # momentum
    f['ema_fast'] = lag1.ewm(span=EMA_FAST_SPAN, adjust=False).mean()
    f['ema_slow'] = lag1.ewm(span=EMA_SLOW_SPAN, adjust=False).mean()
    f['macd'] = f['ema_fast'] - f['ema_slow']
    f['sales_ratio'] = lag1 / (f['ema_slow'] + 1e-6)

    # recent trend (ürün bazında shift: sütunlar ürünlerdir)
    rm = lag1.rolling(RECENT_WINDOW, min_periods=ROLLING_MIN_PERIODS).mean()
    f['recent_trend_4'] = rm - rm.shift(RECENT_SHIFT)

    # zero ratio
    is_zero = (lag1 == 0).astype('float64').where(lag1.notna())
    f['zero_ratio_12'] = is_zero.rolling(ZERO_RATIO_WINDOW, min_periods=ROLLING_MIN_PERIODS).mean()

    # SBC: demand_rate, adi, cv2, sbc_category
    nz = lag1.where(lag1 > 0)
    n_obs = lag1.rolling(SBC_WINDOW, min_periods=1).count()
    n_nz = nz.rolling(SBC_WINDOW, min_periods=1).count()
    f['demand_rate'] = (n_nz / n_obs).where(n_obs > 0)
    f['adi'] = (n_obs / n_nz).where(n_nz >= 2)
    nz_mean = nz.rolling(SBC_WINDOW, min_periods=2).mean()
    nz_std0 = nz.rolling(SBC_WINDOW, min_periods=2).std(ddof=0)
    f['cv2'] = (nz_std0 / nz_mean) ** 2
    adi, cv2 = f['adi'].values, f['cv2'].values
    f['sbc_category'] = np.select(
        [np.isnan(adi) | np.isnan(cv2),
         (adi < SBC_ADI_THRESHOLD) & (cv2 < SBC_CV2_THRESHOLD),
         (adi >= SBC_ADI_THRESHOLD) & (cv2 < SBC_CV2_THRESHOLD),
         (adi < SBC_ADI_THRESHOLD) & (cv2 >= SBC_CV2_THRESHOLD)],
        ['LowData', 'Smooth', 'Intermittent', 'Erratic'], default='Lumpy')

    # hacim: yalnızca satış olan haftalar (gerçek "nonzero")
    obs_ok = lag1.rolling(VOLUME_WINDOW, min_periods=1).count() >= VOLUME_MIN_PERIODS
    f['mean_nonzero_demand'] = nz.rolling(VOLUME_WINDOW, min_periods=1).mean().where(obs_ok)
    f['median_nonzero_demand'] = nz.rolling(VOLUME_WINDOW, min_periods=1).median().where(obs_ok)
    f['q75_nonzero_demand'] = nz.rolling(VOLUME_WINDOW, min_periods=1).quantile(0.75).where(obs_ok)
    f['std_nonzero_demand'] = nz.rolling(VOLUME_WINDOW, min_periods=2).std().where(obs_ok)

    # talepler arası süre: pencere içindeki ardışık pozitif haftaların farkları teleskop toplamıdır
    pos_idx = pd.DataFrame(np.where(lag1.values > 0, t_idx, np.nan), index=wide.index, columns=wide.columns)
    n_win = (lag1 > 0).astype('float64').where(lag1.notna()).rolling(INTER_DEMAND_WINDOW, min_periods=1).sum()
    last_pos = pos_idx.rolling(INTER_DEMAND_WINDOW, min_periods=1).max()
    first_pos = pos_idx.rolling(INTER_DEMAND_WINDOW, min_periods=1).min()
    f['mean_demand_interval'] = ((last_pos - first_pos) / (n_win - 1)).where(n_win >= 3)

    # Croston
    c_d, c_i = _croston(y.values, CROSTON_ALPHA)
    f['croston_alpha_demand'] = pd.DataFrame(c_d, index=wide.index, columns=wide.columns)
    f['croston_alpha_interval'] = pd.DataFrame(c_i, index=wide.index, columns=wide.columns)
    f['croston_forecast'] = f['croston_alpha_demand'] / f['croston_alpha_interval']

    # aday feature'lar
    # pos_idx, lag_1 satırına göre (t) işaretlendiği için gerçek satış haftası t-1'dir
    f['weeks_since_last_demand'] = t_idx - pos_idx.ffill() + 1
    f['history_weeks'] = pd.DataFrame(t_idx - first[None, :], index=wide.index, columns=wide.columns).astype('float64')

    # takvim (ISO haftası)
    week_no = wide.index.isocalendar().week.values.astype('float64')[:, None]
    angle = 2 * np.pi / FOURIER_PERIOD * week_no
    ones = np.ones((1, P))
    f['sin_1'] = pd.DataFrame(np.sin(angle) * ones, index=wide.index, columns=wide.columns)
    f['sin_2'] = pd.DataFrame(np.sin(2 * angle) * ones, index=wide.index, columns=wide.columns)
    f['cos_1'] = pd.DataFrame(np.cos(angle) * ones, index=wide.index, columns=wide.columns)
    f['cos_2'] = pd.DataFrame(np.cos(2 * angle) * ones, index=wide.index, columns=wide.columns)

    # uzun tabloya çevir: yalnızca ürün ömrü içindeki satırlar (ürün, hafta sıralı)
    pi, ti = np.nonzero(life.T)
    out = pd.DataFrame({
        'product': wide.columns.values[pi],
        'week': wide.index.values[ti],
        'quantity': wide.values[ti, pi],
    })
    for name in FEATURES:
        arr = np.asarray(f[name])
        vals = arr[ti, pi]
        if name == 'sbc_category':
            out[name] = pd.Categorical(vals, categories=['Smooth', 'Intermittent', 'Erratic', 'Lumpy', 'LowData'])
        else:
            vals = vals.astype('float64')
            vals[~np.isfinite(vals)] = np.nan
            out[name] = vals.astype('float32')
    return out

In [ ]:
if __name__ == '__main__':
  features = build_features(df_edit)
  features.to_csv('sales_features.csv', index=False, encoding='utf-8')